In [17]:
import pandas as pd

url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet"
columns = ['PULocationID', 'DOLocationID', 'trip_distance', 'total_amount', 'tpep_pickup_datetime']
df = pd.read_parquet(url, columns=columns).head(1000)
df.head()

,PULocationID,DOLocationID,trip_distance,total_amount,tpep_pickup_datetime
0,43,186,1.68,22.15,2025-11-01 00:13:25
1,142,237,2.28,24.94,2025-11-01 00:49:07
2,163,238,2.70,25.62,2025-11-01 00:07:19
3,138,261,12.87,86.14,2025-11-01 00:00:00
4,138,37,8.40,48.65,2025-11-01 00:18:50


In [3]:
from dataclasses import dataclass
import dataclasses

@dataclass
class Ride:
    PULocationID: int
    DOLocationID: int
    trip_distance: float
    total_amount: float
    tpep_pickup_datetime: int  # epoch milliseconds

In [4]:
def ride_from_row(row):
    return Ride(
        PULocationID=int(row['PULocationID']),
        DOLocationID=int(row['DOLocationID']),
        trip_distance=float(row['trip_distance']),
        total_amount=float(row['total_amount']),
        tpep_pickup_datetime=int(row['tpep_pickup_datetime'].timestamp() * 1000),
    )

In [9]:
import json
from kafka import KafkaProducer

server = 'localhost:9092'

topic_name = 'rides'

def ride_seriallizer(ride):
    as_dict = dataclasses.asdict(ride)
    return json.dumps(as_dict).encode('utf-8')

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=ride_seriallizer
)


In [18]:
for _, row in df.iterrows():
    ride = ride_from_row(row)
    producer.send(topic_name, value=ride)
    producer.flush()

In [13]:
producer.send(topic_name, value=ride)
producer.flush()

In [15]:
from kafka import KafkaConsumer

def ride_deserializer(data):
    json_str = data.decode('utf-8')
    ride_dict = json.loads(json_str)
    return Ride(**ride_dict)

consumer = KafkaConsumer(
        topic_name,
        bootstrap_servers=[server],
        auto_offset_reset='earliest',
        group_id='rides-console-v2',
        value_deserializer=ride_deserializer
    )

In [19]:
from datetime import datetime

print(f"Listening to {topic_name}...")

count = 0
for message in consumer:
    ride = message.value
    pickup_dt = datetime.fromtimestamp(ride.tpep_pickup_datetime / 1000)
    print(f"Received: PU={ride.PULocationID}, DO={ride.DOLocationID}, "
          f"distance={ride.trip_distance}, amount=${ride.total_amount:.2f}, "
          f"pickup={pickup_dt}")
    count += 1
    if count >= 10:
        print(f"\n... received {count} messages so far (stopping after 10 for demo)")
        break

consumer.close()

Listening to rides...
Received: PU=43, DO=186, distance=1.68, amount=$22.15, pickup=2025-11-01 07:13:25
Received: PU=142, DO=237, distance=2.28, amount=$24.94, pickup=2025-11-01 07:49:07
Received: PU=163, DO=238, distance=2.7, amount=$25.62, pickup=2025-11-01 07:07:19
Received: PU=138, DO=261, distance=12.87, amount=$86.14, pickup=2025-11-01 07:00:00
Received: PU=138, DO=37, distance=8.4, amount=$48.65, pickup=2025-11-01 07:18:50
Received: PU=90, DO=100, distance=0.85, amount=$16.45, pickup=2025-11-01 07:21:11
Received: PU=142, DO=170, distance=3.01, amount=$25.85, pickup=2025-11-01 07:07:31
Received: PU=237, DO=144, distance=3.82, amount=$57.54, pickup=2025-11-01 07:46:52
Received: PU=162, DO=161, distance=0.89, amount=$12.95, pickup=2025-11-01 07:56:59
Received: PU=234, DO=162, distance=2.28, amount=$38.68, pickup=2025-11-01 07:10:43

... received 10 messages so far (stopping after 10 for demo)
